In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
import seaborn as sns
from rapidfuzz import fuzz, process
from sklearn.model_selection import cross_validate
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

from scholarlm.utils import (
    load_and_process_results,
    match_datasets,
    matching_precision_recall,
    get_filenames_in_directory,
    fit_temperature,
    apply_temperature,
    fit_temperature_from_probs,
    apply_temperature_from_probs,
)
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2

INFO 03-18 09:32:01 [__init__.py:220] No platform detected, vLLM is running on UnspecifiedPlatform
WARNING 03-18 09:32:14 [_custom_ops.py:20] Failed to import from vllm._C with ImportError('libcuda.so.1: cannot open shared object file: No such file or directory')


### Ground Truth Dataset

In [3]:
# ---------------------------------
# Load from ground truth dataset
# ---------------------------------

# Directory
with open(os.path.join("../data/pond/directory.json"), "r") as f:
    paper_info = json.load(f)

paper_subset = [
    'physical_and_chemical_limnological',
    'physical-chemical_influences',
    'prairie_wetland',
    'net_heterotrophy',
    'habitat_characteristics',
    'biodiversity_of_constructed',
    'fish_production_in_lakes',
    'long-term_stability',
    'diversity_of_macroinvertebrates',
    'impact_of_macrophytes'
]

paper_info = {k:v for k,v in paper_info.items() if k in paper_subset}
registered_titles = [entry['title'] for entry in paper_info.values()]
registered_titles.sort()

ground_truth_df = pd.read_csv("../data/pond/pond_data_corrected.csv", encoding_errors='ignore', index_col = 0)
ground_truth_df = ground_truth_df.loc[ground_truth_df.title.isin(registered_titles)]
ground_truth_df = ground_truth_df.reset_index(drop=True)

### Elicit Extracted Dataset

In [ ]:
'''
# Parse from elicit CSV file:
import json

def parse_json_string(json_string, default=None):
    """
    Safely parse a JSON string into a dictionary.
    
    Args:
        json_string: The JSON string to parse
        default: The default value to return if parsing fails (default: None)
    
    Returns:
        Dictionary if parsing succeeds, otherwise the default value
    """
    try:
        result = json.loads(json_string.replace('None', 'null'))
        return result
    #except json.JSONDecodeError as e:
    except:
        print('--------------------------------------------')
        print('JSON')
        print(json_string)
        print()
        return default if default is not None else {}
    

elicit_df = pd.read_csv("../data/12_10_25/elicit_extraction2.csv")

extraction_features = [
    'surface_area',
    'max_depth',
    'vegetation_cover',
    'ph',
    'tn',
    'tp',
    'chla'
]
result_dict = []
for i, row in elicit_df.iterrows():
    fname = row['Filename']
    paper_code = fname.replace('.pdf','')
    paper_metadata = paper_info[paper_code]

    for f in extraction_features:
        response = row[f]
        response_dict = parse_json_string(response, default = {'items': []})
        for it in response_dict['items']:
            datapoint = paper_metadata | it | {'measurement': f}
            result_dict.append(datapoint)

output_file = "../data/12_10_25/pond_elicit.json"
with open(output_file, "w") as f:
    json.dump(result_dict, f, indent=4, ensure_ascii=False)
'''

In [28]:
json_path = "../data/experiments/2025_12_10/pond_elicit.json"
with open(json_path, "r") as f:
    records = json.load(f)

In [29]:
records

[{'title': 'net heterotrophy in small danish lakes: a widespread feature over gradients in trophic status and land cover',
  'author': 'sand-jensen',
  'year': 2009,
  'name': 'Agertoften',
  'date': 'November 2000 to November 2001',
  'location': 'North Zealand, Denmark',
  'ecosystem': 'lake',
  'value': '6,061',
  'units': 'm^2',
  'measurement': 'surface_area'},
 {'title': 'net heterotrophy in small danish lakes: a widespread feature over gradients in trophic status and land cover',
  'author': 'sand-jensen',
  'year': 2009,
  'name': 'Badstuedam',
  'date': 'November 2000 to November 2001',
  'location': 'North Zealand, Denmark',
  'ecosystem': 'lake',
  'value': '28,039',
  'units': 'm^2',
  'measurement': 'surface_area'},
 {'title': 'net heterotrophy in small danish lakes: a widespread feature over gradients in trophic status and land cover',
  'author': 'sand-jensen',
  'year': 2009,
  'name': 'Bendstrup Sø',
  'date': 'November 2000 to November 2001',
  'location': 'North Zeal

In [36]:
# ---------------------------------
# Load experiment results
# ---------------------------------

experiment_data_path = "../data/experiments/2025_12_10/pond_elicit_judged.json"

unit_conversion_table = {
    'max_depth': {"cm": 0.01, "feet": 0.3048, "km": 1000, "m": 1},
    'surface_area': {"km^2": 1e6, "ha": 1e4, "mi^2": 2.59e6, "m^2": 1, "acres": 4046.86},
    'vegetation_cover': {"percent": 1.0, "fraction": 100.0},
    'tn': {"mg/L": 1000.0, "µg/L": 1.0, "μmol/L": 14.01, "ppm": 1000.0, "ppb": 1.0},
    'tp': {"mg/L": 1000.0, "µg/L": 1.0, "μmol/L": 30.97, "ppm": 1000.0, "ppb": 1.0},
    'chla': {"mg/L": 1000.0, "µg/L": 1.0, "mg/m^3": 1.0},
    'ph': {},
    'latitude': {},
    'longitude': {}
}

attribute_types = {
    'max_depth': float,
    'surface_area': float,
    'vegetation_cover': float,
    'tn': float,
    'tp': float,
    'chla': float,
    'ph': float,
    'latitude': float,
    'longitude': float
}

# NOTE: some of these things you should get rid of in your extraction process!
drop_keys = None
drop_attrs = ['latitude', 'longitude']

extracted_df = load_and_process_results(
    json_path=experiment_data_path,
    unit_conversion_table=unit_conversion_table,
    attribute_types=attribute_types,
    drop_keys=drop_keys,
    drop_attrs=drop_attrs,
    attribute_col="measurement",
    value_col="value",
    unit_col="units",
    out_col="processed_value",
    strict_unit_conversion=False
)

# NOTE you need to change this to 'attribute'
extracted_df.rename(columns={"feature": "attribute", "measurement":"attribute"}, inplace=True)
extracted_df.sort_values(by=["title", "attribute"], inplace=True)
extracted_df.reset_index(drop=True, inplace=True)

### Match with ground truth

In [34]:
# Set of attributes which must be strictly equivalent to create a match
strict_matching = {
    "title": "title", # name in the ground truth dataset : name in the extracted dataset
    "attribute": "attribute",
    "value": "processed_value"
}

# Set of attributes which should be 
# compared by a fuzzy matching (roughly similar) to create a match.
fuzzy_matching = {
    "name": "name",
    "location": "location",
    "ecosystem": "ecosystem",
}

# This can take a while to run if you have a lot of data, 
# since it compares every extracted row to every ground truth row.
matching, matching_recall, matching_precision = matching_precision_recall(
    ground_truth_df,
    extracted_df,
    strict_matching=strict_matching,
    fuzzy_matching=fuzzy_matching,
    fuzzy_threshold = 1/3
)

print(f"Recall: {matching_recall:.4f}")

Recall: 0.1684


In [35]:
print(f"Precision: {matching_precision:.4f}")

Precision: 0.6692
